# Module 2 · Sampling Parameters
Control how the model generates text: temperature, top_k, top_p, max_output_tokens, stop_sequences.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # picks up modules/.env symlink
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemma-4-31b-it'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                # parse suggested retry delay from error (e.g. 'retryDelay': '30s')
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('\u2705 Setup complete. Model:', MODEL, '| Retry-on-429: enabled')


✅ Setup complete. Model: gemma-4-31b-it | Retry-on-429: enabled


### 🔌 API Key Test

In [3]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


---
## 4. Core Sampling Parameters

```
All vocabulary tokens (~32k)
         │
    top_k filter  ──→  keep only top K tokens by probability
         │
    top_p filter  ──→  keep smallest set whose cumulative prob ≥ p
         │
  temperature     ──→  sharpen (< 1) or flatten (> 1) the distribution
         │
      SAMPLE  ──→  pick one token  ──→  append  ──→  repeat
```
The filters stack: `top_k` first, then `top_p`, then `temperature` reshapes what's left.

### 4a. `temperature` — creativity dial

| Value | Behaviour |
|---|---|
| `0.0` | Always pick the highest-probability token (deterministic) |
| `0.7` | Balanced — good default |
| `1.0` | Raw model probabilities, no reshaping |
| `2.0` | Distribution nearly flat — very random |

**Analogy:** Job candidates ranked by score. `temperature=0` → always hire rank #1. `temperature=2` → shuffle the entire list randomly.

In [4]:
prompt = "Continue this story in ONE sentence only: 'The robot opened the door and saw'"

for temp in [0.0, 0.7, 1.5]:
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=temp)
    )
    print(f"temperature={temp}: {get_text(r)}")
    print()

temperature=0.0: The robot opened the door and saw a mirror reflecting a human face, the only remnant of a species it had accidentally erased.



temperature=0.7: The robot opened the door and saw a mirror reflecting a version of itself that had finally learned how to dream.



temperature=1.5: a mirror that reflected a human face, though it knew its body was made of cold, polished steel.



In [5]:
# temperature=0 → near-identical results across runs
print("Same prompt × 3 at temperature=0:")
for i in range(3):
    r = client.models.generate_content(
        model=MODEL, contents="What is 17 × 8? Reply with only the number.",
        config=cfg(temperature=0.0)
    )
    print(f"  Run {i+1}: {get_text(r)}")

Same prompt × 3 at temperature=0:


  Run 1: 136


  Run 2: 136


  Run 3: 136


### 4b. `top_k` — vocabulary hard limit

Before sampling, **discard all but the top-k tokens** by probability.

| Value | Effect |
|---|---|
| `1` | Greedy — always pick the most probable token |
| `10` | Only consider the 10 most likely tokens |
| `40` | Typical default |

**Analogy:** Only put the top-k most-likely words in a hat before drawing.

In [6]:
prompt = "Reply with ONLY a single color name, nothing else:"
for k in [1, 5, 100]:
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=1.0, top_k=k)
    )
    print(f"top_k={k:>3}: {get_text(r)}")

top_k=  1: Blue


top_k=  5: Blue


top_k=100: Blue


In [7]:
# top_k=1 → same token every time (greedy)
print("top_k=1 repeated 3 times (expect same answer):")
for _ in range(3):
    r = client.models.generate_content(
        model=MODEL,
        contents="The most widely-used programming language is (one word only):",
        config=cfg(top_k=1, temperature=0.0)
    )
    print(" ", get_text(r))

top_k=1 repeated 3 times (expect same answer):


  JavaScript


  JavaScript


  JavaScript


### 4c. `top_p` — nucleus sampling

Keep the **smallest set of tokens whose cumulative probability ≥ p**.

```
"the"   → 55%  cumulative=55%  keep
"a"     → 20%  cumulative=75%  keep
"this"  → 10%  cumulative=85%  keep
"that"  →  8%  cumulative=93%  ← STOP (crossed 0.9)
```

| Value | Effect |
|---|---|
| `0.1` | Ultra-narrow: only top tokens |
| `0.9` | Common default |
| `1.0` | No filtering |

In [8]:
prompt = "Give ONE synonym for 'happy' (single word only):"
for p in [0.05, 0.5, 1.0]:
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=1.0, top_p=p)
    )
    print(f"top_p={p}: {get_text(r)}")

top_p=0.05: Joyful


top_p=0.5: Joyful


top_p=1.0: Joyful


### 4d. `max_output_tokens` — length hard cap

Stops generation after this many tokens. It's a **budget**, not a target.

> ⚠️ **Gemma 4 note:** Because Gemma 4 uses thinking tokens internally, you need a higher `max_output_tokens` than you might expect. `DEFAULT_MAX=2048` is used throughout this notebook.

In [9]:
prompt = "Explain what machine learning is."
for limit in [300, 700, 2048]:
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=0.5, max_output_tokens=limit)
    )
    finish = r.candidates[0].finish_reason
    used = r.usage_metadata.candidates_token_count or 0
    thoughts = getattr(r.usage_metadata, 'thoughts_token_count', 0) or 0
    text = get_text(r)
    print(f"--- max={limit} (thinking={thoughts}, answer={used}, finish={finish.name}) ---")
    print(text[:200], "..." if len(text) > 200 else "")
    print()

--- max=300 (thinking=297, answer=0, finish=MAX_TOKENS) ---
 



--- max=700 (thinking=502, answer=193, finish=MAX_TOKENS) ---
At its simplest level, **Machine Learning (ML)** is a field of artificial intelligence (AI) that gives computers the ability to learn without being explicitly programmed.

Here is a detailed breakdown ...



--- max=2048 (thinking=491, answer=928, finish=STOP) ---
At its simplest level, **Machine Learning (ML)** is a field of artificial intelligence (AI) that gives computers the ability to learn without being explicitly programmed.

To understand this, it helps ...



| `finish_reason` | Meaning |
|---|---|
| `STOP` | Model finished naturally |
| `MAX_TOKENS` | Cut off — increase `max_output_tokens` |
| `SAFETY` | Blocked by safety filters |

### 4e. `stop_sequences` — custom stop strings

Stop generation when one of these strings is produced. Great for structured extraction.

In [10]:
prompt = "List three programming languages, one per line, numbered 1. 2. 3.:"

r_full = client.models.generate_content(
    model=MODEL, contents=prompt, config=cfg(temperature=0.0)
)
print("Without stop_sequences:")
print(get_text(r_full))
print()

r_stopped = client.models.generate_content(
    model=MODEL, contents=prompt,
    config=cfg(temperature=0.0, stop_sequences=["3."])
)
print("With stop_sequences=['3.'] — stops before item 3:")
print(repr(get_text(r_stopped)))

Without stop_sequences:
1. Python
2. JavaScript
3. Java



With stop_sequences=['3.'] — stops before item 3:
'1. Python\n2. JavaScript'


In [11]:
# Practical use: extract just the country name, stop before any explanation
r = client.models.generate_content(
    model=MODEL,
    contents="What country is Paris in? Answer with ONLY the country name, nothing else.",
    config=cfg(temperature=0.0, stop_sequences=[".", "\n"])
)
print("Extracted country:", get_text(r))

Extracted country: France


### 4f. Parameter Quick-Reference

| Task | temperature | top_k | top_p |
|---|---|---|---|
| Factual Q&A / extraction | 0.0–0.2 | 1–10 | 0.1–0.5 |
| Balanced chat | 0.7 | 40 | 0.9 |
| Creative writing | 1.0–1.5 | 40–100 | 0.95–1.0 |
| Brainstorming | 1.5–2.0 | 100+ | 1.0 |